# YOLOv8 vs RetinaNet 비교 분석

## 전체 흐름
```
STEP 1. 라이브러리 설치 및 임포트
STEP 2. KITTI → YOLO 형식 변환
STEP 3. YOLOv8 학습
STEP 4. YOLOv8 추론 및 시각화
STEP 5. RetinaNet vs YOLOv8 비교 분석
STEP 6. 자율주행 보조 시스템 적용
```

## RetinaNet vs YOLOv8 핵심 차이
| 항목 | RetinaNet | YOLOv8 |
|------|-----------|--------|
| 앵커 방식 | 앵커 기반 (92,000개) | 앵커 프리 |
| 백본 | ResNet50 + FPN | CSPDarknet + FPN |
| Loss | Focal Loss + SmoothL1 | BCE + CIoU |
| 코드 | 직접 구현 | ultralytics 라이브러리 |
| 속도 | 보통 | 빠름 |

## STEP 1. 라이브러리 설치 및 임포트

In [1]:
# YOLOv8 라이브러리 설치
# ultralytics: YOLOv8을 포함한 YOLO 시리즈를 쉽게 사용할 수 있는 공식 라이브러리
!pip install ultralytics --quiet
print('설치 완료 ✅')

설치 완료 ✅


In [2]:
import os
import shutil
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw, ImageFont
from tqdm import tqdm
import torch
from ultralytics import YOLO   # YOLOv8 라이브러리

# 경로 설정
HOME      = os.getenv('HOME')
IMG_DIR   = os.path.join(HOME, 'work/object_detection/data/Kitti/raw/training/image_2')
LABEL_DIR = os.path.join(HOME, 'work/object_detection/data/Kitti/raw/training/label_2')
YOLO_DIR  = os.path.join(HOME, 'work/object_detection/data/kitti_yolo')   # 변환된 데이터 저장 위치
YOLO_RUN_DIR = os.path.join(HOME, 'work/object_detection/yolo_runs')      # 학습 결과 저장 위치
RETINA_CKPT  = os.path.join(HOME, 'work/object_detection/checkpoints/best.pth')  # RetinaNet 체크포인트

# KITTI 클래스 목록 (RetinaNet과 동일하게 유지)
CLASSES = ['Car', 'Van', 'Truck', 'Pedestrian',
           'Person_sitting', 'Cyclist', 'Tram', 'Misc']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

# GPU 확인
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'사용 디바이스: {device}')
print(f'클래스 수: {len(CLASSES)}')
print('임포트 완료 ✅')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/jovyan/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
사용 디바이스: cuda
클래스 수: 8
임포트 완료 ✅


## STEP 2. KITTI → YOLO 형식 변환

YOLOv8은 RetinaNet과 라벨 형식이 다름.

```
KITTI 형식: Car 0.0 0 1.57 600 150 860 370 ...  (픽셀 절대 좌표)
                             ↑   ↑   ↑   ↑
                           x_min y_min x_max y_max

YOLO 형식:  0 0.547 0.406 0.159 0.373  (0~1 정규화된 중심점+크기)
            ↑   ↑     ↑     ↑     ↑
          cls  cx    cy    w     h
```

변환 공식:
```
cx = (x_min + x_max) / 2 / image_width
cy = (y_min + y_max) / 2 / image_height
w  = (x_max - x_min) / image_width
h  = (y_max - y_min) / image_height
```

In [3]:
def kitti_to_yolo_label(label_path, img_width, img_height):
    """
    KITTI 라벨 파일 1개를 YOLO 형식 문자열로 변환
    
    Args:
        label_path : KITTI .txt 라벨 파일 경로
        img_width  : 이미지 너비 (픽셀)
        img_height : 이미지 높이 (픽셀)
    Returns:
        YOLO 형식 라벨 문자열 리스트
    """
    lines_out = []
    
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 8:
                continue
            
            class_name = parts[0]
            
            # DontCare 및 미사용 클래스 제외
            if class_name not in CLASS_TO_IDX or class_name == 'DontCare':
                continue
            
            # KITTI bbox 좌표 (픽셀)
            x_min = float(parts[4])
            y_min = float(parts[5])
            x_max = float(parts[6])
            y_max = float(parts[7])
            
            # 유효하지 않은 bbox 제외
            if x_max <= x_min or y_max <= y_min:
                continue
            
            # YOLO 형식으로 변환 (0~1 정규화)
            cx = (x_min + x_max) / 2 / img_width
            cy = (y_min + y_max) / 2 / img_height
            w  = (x_max - x_min) / img_width
            h  = (y_max - y_min) / img_height
            
            # 범위 클리핑 (0~1 벗어나지 않도록)
            cx = min(max(cx, 0), 1)
            cy = min(max(cy, 0), 1)
            w  = min(max(w, 0), 1)
            h  = min(max(h, 0), 1)
            
            cls_id = CLASS_TO_IDX[class_name]
            lines_out.append(f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    
    return lines_out


def convert_kitti_to_yolo(img_dir, label_dir, yolo_dir, train_ratio=0.9):
    """
    KITTI 전체 데이터셋을 YOLO 형식으로 변환
    
    폴더 구조 생성:
        kitti_yolo/
          images/train/  ← 학습 이미지
          images/val/    ← 검증 이미지
          labels/train/  ← 학습 라벨 (YOLO 형식)
          labels/val/    ← 검증 라벨 (YOLO 형식)
    """
    # 폴더 생성
    for split in ['train', 'val']:
        os.makedirs(os.path.join(yolo_dir, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(yolo_dir, 'labels', split), exist_ok=True)
    
    # 이미지 파일 목록
    img_files = sorted([f for f in os.listdir(img_dir)
                        if f.endswith('.png') or f.endswith('.jpg')])
    
    # train/val 분할 (90% / 10%)
    split_idx   = int(len(img_files) * train_ratio)
    train_files = img_files[:split_idx]
    val_files   = img_files[split_idx:]
    print(f'Train: {len(train_files)}개 / Val: {len(val_files)}개')
    
    def process(files, split):
        converted = 0
        skipped   = 0
        for fname in tqdm(files, desc=f'{split} 변환 중'):
            img_path   = os.path.join(img_dir, fname)
            label_path = os.path.join(label_dir,
                         fname.replace('.png', '.txt').replace('.jpg', '.txt'))
            
            if not os.path.exists(label_path):
                skipped += 1
                continue
            
            img    = Image.open(img_path)
            W, H   = img.size
            labels = kitti_to_yolo_label(label_path, W, H)
            
            if not labels:   # 유효한 객체 없으면 스킵
                skipped += 1
                continue
            
            # 이미지 복사
            shutil.copy(img_path, os.path.join(yolo_dir, 'images', split, fname))
            
            # 라벨 저장
            label_fname = fname.replace('.png', '.txt').replace('.jpg', '.txt')
            with open(os.path.join(yolo_dir, 'labels', split, label_fname), 'w') as f:
                f.write('\n'.join(labels))
            converted += 1
        
        print(f'  {split}: 변환 {converted}개 / 스킵 {skipped}개')
    
    process(train_files, 'train')
    process(val_files,   'val')
    print('\n변환 완료 ✅')


# 실행
convert_kitti_to_yolo(IMG_DIR, LABEL_DIR, YOLO_DIR)

Train: 6732개 / Val: 749개


train 변환 중: 100%|██████████| 6732/6732 [00:21<00:00, 314.78it/s] 


  train: 변환 6732개 / 스킵 0개


val 변환 중: 100%|██████████| 749/749 [00:03<00:00, 200.91it/s]

  val: 변환 749개 / 스킵 0개

변환 완료 ✅


## STEP 3. YAML 설정 파일 생성

YOLOv8은 데이터셋 경로와 클래스 정보를 YAML 파일로 관리

In [4]:
yaml_content = f"""# KITTI 데이터셋 설정
path: {YOLO_DIR}     # 데이터셋 루트 경로
train: images/train  # 학습 이미지 경로 (path 기준 상대경로)
val: images/val      # 검증 이미지 경로

# 클래스 수와 이름 (RetinaNet과 동일)
nc: 8
names:
  0: Car
  1: Van
  2: Truck
  3: Pedestrian
  4: Person_sitting
  5: Cyclist
  6: Tram
  7: Misc
"""

yaml_path = os.path.join(YOLO_DIR, 'kitti.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f'YAML 저장 완료: {yaml_path} ✅')
print(yaml_content)

YAML 저장 완료: /home/jovyan/work/object_detection/data/kitti_yolo/kitti.yaml ✅
# KITTI 데이터셋 설정
path: /home/jovyan/work/object_detection/data/kitti_yolo     # 데이터셋 루트 경로
train: images/train  # 학습 이미지 경로 (path 기준 상대경로)
val: images/val      # 검증 이미지 경로

# 클래스 수와 이름 (RetinaNet과 동일)
nc: 8
names:
  0: Car
  1: Van
  2: Truck
  3: Pedestrian
  4: Person_sitting
  5: Cyclist
  6: Tram
  7: Misc



## STEP 4. YOLOv8 학습

### RetinaNet vs YOLOv8 학습 방식 비교
```
RetinaNet: 직접 구현
  - AnchorBox 생성 → LabelEncoder → FocalLoss → 역전파 → save_checkpoint

YOLOv8: ultralytics 라이브러리
  - model.train() 한 줄로 전체 학습 처리
  - 앵커 프리, 자동 mAP 계산, 자동 체크포인트 저장
```

In [5]:
# YOLOv8s 모델 로드
# yolov8s.pt: ImageNet 사전학습된 YOLOv8 small 모델
# n(nano) < s(small) < m(medium) < l(large) < x(extra-large)
yolo_model = YOLO('yolov8s.pt')
print('YOLOv8s 모델 로드 완료 ✅')

YOLOv8s 모델 로드 완료 ✅


In [6]:
# YOLOv8 학습 실행
# RetinaNet과 비교를 위해 동일한 epochs=5 사용
results = yolo_model.train(
    data=yaml_path,          # YAML 설정 파일
    epochs=5,                # RetinaNet과 동일한 epochs
    imgsz=640,               # 입력 이미지 크기 (정사각형)
    batch=4,                 # 배치 크기 (OOM 방지)
    name='kitti_yolov8s',    # 실험 이름
    project=YOLO_RUN_DIR,    # 저장 경로
    device=0 if device == 'cuda' else 'cpu',
    workers=0,               # 데이터 로딩 프로세스 수
    patience=3,              # Early stopping (3 에폭 개선 없으면 중단)
    save=True,               # 체크포인트 자동 저장
    val=True,                # 학습 중 검증 수행
    verbose=True,            # 학습 과정 출력
)

print('\nYOLOv8 학습 완료 ✅')
print(f'결과 저장 위치: {YOLO_RUN_DIR}/kitti_yolov8s/')

Ultralytics 8.4.46 🚀 Python-3.12.11 torch-2.7.1+cu118 CUDA:0 (Tesla T4, 14931MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jovyan/work/object_detection/data/kitti_yolo/kitti.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=kitti_yolov8s, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto

## STEP 5. YOLOv8 성능 평가

In [9]:
# 학습된 best 모델로 검증셋 평가
best_model_path = os.path.join(YOLO_RUN_DIR, 'kitti_yolov8s/weights/best.pt')

if os.path.exists(best_model_path):
    best_model = YOLO(best_model_path)
    
    val_results = best_model.val(
        data=yaml_path,
        split='val',
        verbose=True,
        workers=0
    )
    
    # 결과 저장
    yolo_map50   = float(val_results.box.map50)
    yolo_map5095 = float(val_results.box.map)
    
    print(f'\n=== YOLOv8 평가 결과 ===')
    print(f'mAP@0.5     : {yolo_map50:.4f}')
    print(f'mAP@0.5:0.95: {yolo_map5095:.4f}')

else:
    print(f'모델 파일 없음: {best_model_path}')
    print('학습을 먼저 완료해주세요')

Ultralytics 8.4.46 🚀 Python-3.12.11 torch-2.7.1+cu118 CUDA:0 (Tesla T4, 14931MiB)
Model summary (fused): 73 layers, 11,128,680 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3864.1±596.7 MB/s, size: 849.9 KB)
val: Scanning /home/jovyan/work/object_detection/data/kitti_yolo/labels/val.cache... 749 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 749/749 224.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 3.1it/s 15.2s0.3s
                   all        749       3942      0.865      0.652      0.774      0.518
                   Car        664       2845      0.914      0.864      0.934      0.712
                   Van        214        264       0.85      0.625      0.791      0.589
                 Truck         96        104       0.91      0.808      0.929       0.71
            Pedestrian        172        400      0.909      0.547      0.725      0.378
        P

## STEP 6. YOLOv8 추론 속도(FPS) 측정

RetinaNet과 공정한 비교를 위해 동일한 이미지로 FPS를 측정.

In [10]:
def measure_fps(model, img_dir, num_images=100):
    """
    추론 속도(FPS) 측정
    
    FPS = 초당 처리 이미지 수
    FPS가 높을수록 빠른 모델
    
    Args:
        model      : YOLO 모델
        img_dir    : 테스트 이미지 디렉토리
        num_images : 측정에 사용할 이미지 수
    """
    img_files = [os.path.join(img_dir, f)
                 for f in os.listdir(img_dir)
                 if f.endswith('.png') or f.endswith('.jpg')][:num_images]
    
    # 워밍업 (처음 몇 장은 GPU 초기화로 느림)
    print('워밍업 중...')
    for img_path in img_files[:5]:
        model.predict(img_path, verbose=False)
    
    # 실제 FPS 측정
    print(f'{len(img_files)}장 이미지로 FPS 측정 중...')
    start = time.time()
    for img_path in img_files:
        model.predict(img_path, verbose=False)
    elapsed = time.time() - start
    
    fps = len(img_files) / elapsed
    print(f'\n처리 시간: {elapsed:.2f}초')
    print(f'FPS: {fps:.2f}')
    return fps


# YOLOv8 FPS 측정
val_img_dir = os.path.join(YOLO_DIR, 'images/val')
if os.path.exists(best_model_path):
    best_model = YOLO(best_model_path)
    yolo_fps = measure_fps(best_model, val_img_dir, num_images=100)
else:
    print('학습 완료 후 실행해주세요')

워밍업 중...
100장 이미지로 FPS 측정 중...

처리 시간: 2.16초
FPS: 46.25


## STEP 7. YOLOv8 결과 시각화 (20장)

In [11]:
# 클래스별 색상 (RetinaNet과 동일하게 유지 → 비교 용이)
CLASS_COLORS = {
    'Car':            (255, 0, 0),
    'Van':            (255, 128, 0),
    'Truck':          (255, 255, 0),
    'Pedestrian':     (0, 255, 0),     # 초록 - 자율주행 핵심!
    'Person_sitting': (0, 255, 128),
    'Cyclist':        (0, 128, 255),
    'Tram':           (128, 0, 255),
    'Misc':           (255, 0, 255)
}

def visualize_yolo_result(img_path, result, output_path, min_conf=0.3):
    """
    YOLOv8 추론 결과를 이미지에 시각화
    
    Args:
        img_path    : 원본 이미지 경로
        result      : YOLO predict() 반환값
        output_path : 결과 저장 경로
        min_conf    : 최소 신뢰도 임계값
    """
    image = Image.open(img_path).convert('RGB')
    draw  = ImageDraw.Draw(image)
    
    try:
        font = ImageFont.truetype('malgun.ttf', 18)
    except:
        font = ImageFont.load_default()
    
    boxes   = result.boxes
    det_count = 0
    
    if boxes is not None:
        for box in boxes:
            conf = float(box.conf[0])       # 신뢰도 점수
            cls  = int(box.cls[0])          # 클래스 인덱스
            
            if conf < min_conf:
                continue
            
            # bbox 좌표 (xyxy 형식 - 픽셀)
            x1, y1, x2, y2 = map(float, box.xyxy[0])
            class_name = CLASSES[cls]
            color = CLASS_COLORS.get(class_name, (255, 255, 255))
            
            # bbox 그리기
            draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
            
            # 라벨 텍스트
            label = f'{class_name} {conf:.2f}'
            draw.rectangle([x1, y1-22, x1+len(label)*9, y1], fill=color)
            draw.text((x1+2, y1-22), label, fill=(255,255,255), font=font)
            det_count += 1
    
    image.save(output_path)
    return det_count


# 결과 저장 폴더
os.makedirs('result_yolo', exist_ok=True)

if os.path.exists(best_model_path):
    best_model = YOLO(best_model_path)
    
    # 검증 이미지에서 20장 랜덤 선택
    val_imgs = [os.path.join(val_img_dir, f)
                for f in os.listdir(val_img_dir)
                if f.endswith('.png') or f.endswith('.jpg')]
    selected = random.sample(val_imgs, min(20, len(val_imgs)))
    
    for i, img_path in enumerate(selected):
        output_path = f'result_yolo/result_{i+1:02d}.png'
        
        # 추론
        results = best_model.predict(
            img_path,
            conf=0.3,        # 신뢰도 임계값
            iou=0.5,         # NMS IoU 임계값
            verbose=False
        )
        
        count = visualize_yolo_result(img_path, results[0], output_path)
        print(f'[{i+1:02d}/20] {os.path.basename(img_path)} → {count}개 탐지')
    
    print('\nYOLOv8 시각화 완료 ✅  (result_yolo/ 폴더 확인)')
else:
    print('학습 완료 후 실행해주세요')

[01/20] 007208.png → 9개 탐지
[02/20] 007378.png → 7개 탐지
[03/20] 007172.png → 3개 탐지
[04/20] 007261.png → 5개 탐지
[05/20] 006971.png → 5개 탐지
[06/20] 006958.png → 5개 탐지
[07/20] 007232.png → 5개 탐지
[08/20] 006814.png → 10개 탐지
[09/20] 007085.png → 5개 탐지
[10/20] 007326.png → 3개 탐지
[11/20] 006780.png → 11개 탐지
[12/20] 007130.png → 3개 탐지
[13/20] 006820.png → 10개 탐지
[14/20] 007113.png → 1개 탐지
[15/20] 006783.png → 12개 탐지
[16/20] 006948.png → 9개 탐지
[17/20] 006789.png → 11개 탐지
[18/20] 007088.png → 5개 탐지
[19/20] 007452.png → 5개 탐지
[20/20] 006900.png → 2개 탐지

YOLOv8 시각화 완료 ✅  (result_yolo/ 폴더 확인)


## STEP 8. RetinaNet vs YOLOv8 비교 분석

⚠️ RetinaNet 학습이 완료된 후 실행하세요!

아래 셀에서 RetinaNet 결과값을 직접 입력해주세요.

In [19]:
# =====================================================
# RetinaNet 결과값 직접 입력 (학습 완료 후)
# =====================================================
RETINA_MAP50   = 0.5642   # ← RetinaNet mAP@0.5 입력
RETINA_ACC     = 96.47   # ← RetinaNet 정확도(%) 입력
RETINA_FPS     = 12.01   # ← RetinaNet FPS 입력

# YOLOv8 결과값 (STEP 5, 6에서 자동 측정)
YOLO_MAP50     = yolo_map50 if 'yolo_map50' in dir() else 0.0
YOLO_ACC       = 0.0   # ← YOLOv8 정확도(%) 입력
YOLO_FPS       = yolo_fps   if 'yolo_fps'   in dir() else 0.0

# 비교 결과 출력
print('=' * 55)
print(f'{"항목":<20} {"RetinaNet":>15} {"YOLOv8s":>15}')
print('=' * 55)
print(f'{"mAP@0.5":<20} {RETINA_MAP50:>15.4f} {YOLO_MAP50:>15.4f}')
print(f'{"정확도 (%)":<20} {RETINA_ACC:>15.2f} {YOLO_ACC:>15.2f}')
print(f'{"FPS":<20} {RETINA_FPS:>15.2f} {YOLO_FPS:>15.2f}')
print('=' * 55)

# 가설 검증
print('\n=== 가설 검증 ===')
if YOLO_MAP50 > 0 and RETINA_MAP50 > 0:
    if YOLO_FPS > RETINA_FPS:
        print('✅ 가설 1 검증: YOLOv8이 RetinaNet보다 빠름')
    else:
        print('❌ 가설 1 반증: RetinaNet이 YOLOv8보다 빠름 (예상과 다름)')
    
    if RETINA_MAP50 > YOLO_MAP50:
        print('✅ 가설 1 검증: RetinaNet이 YOLOv8보다 mAP 높음')
    else:
        print('❌ 가설 1 반증: YOLOv8이 RetinaNet보다 mAP 높음 (예상과 다름)')
else:
    print('두 모델 학습 완료 후 결과값을 입력해주세요')

항목                         RetinaNet         YOLOv8s
mAP@0.5                       0.5642          0.7740
정확도 (%)                        96.47            0.00
FPS                            12.01           46.25

=== 가설 검증 ===
✅ 가설 1 검증: YOLOv8이 RetinaNet보다 빠름
❌ 가설 1 반증: YOLOv8이 RetinaNet보다 mAP 높음 (예상과 다름)


In [20]:
# 비교 시각화 차트
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('RetinaNet vs YOLOv8 비교 분석', fontsize=16, fontweight='bold')

models  = ['RetinaNet\n(ResNet50)', 'YOLOv8s\n(CSPDarknet)']
colors  = ['#2E75B6', '#E67E22']
metrics = [
    ('mAP@0.5',    [RETINA_MAP50, YOLO_MAP50]),
    ('정확도 (%)', [RETINA_ACC,   YOLO_ACC]),
    ('FPS',        [RETINA_FPS,   YOLO_FPS]),
]

for ax, (title, values) in zip(axes, metrics):
    bars = ax.bar(models, values, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.set_ylim(0, max(values) * 1.3 if max(values) > 0 else 1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # 값 표시
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.02,
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('차트 저장 완료: comparison_chart.png ✅')

/tmp/ipykernel_2258/15795873.py:25: UserWarning: Glyph 51221 (\N{HANGUL SYLLABLE JEONG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2258/15795873.py:25: UserWarning: Glyph 54869 (\N{HANGUL SYLLABLE HWAG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2258/15795873.py:25: UserWarning: Glyph 46020 (\N{HANGUL SYLLABLE DO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2258/15795873.py:25: UserWarning: Glyph 48708 (\N{HANGUL SYLLABLE BI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2258/15795873.py:25: UserWarning: Glyph 44368 (\N{HANGUL SYLLABLE GYO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2258/15795873.py:25: UserWarning: Glyph 48516 (\N{HANGUL SYLLABLE BUN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_2258/15795873.py:25: UserWarning: Glyph 49437 (\N{HANGUL SYLLABLE SEOG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp

<Figure size 1500x500 with 3 Axes>

차트 저장 완료: comparison_chart.png ✅


## STEP 9. YOLOv8 기반 자율주행 보조 시스템

In [5]:
import os
from ultralytics import YOLO

HOME = os.getenv('HOME')

# 경로 설정
best_model_path = os.path.join(HOME, 'work/object_detection/yolo_runs/kitti_yolov8s/weights/best.pt')
YOLO_DIR        = os.path.join(HOME, 'work/object_detection/data/kitti_yolo')
CLASSES = ['Car', 'Van', 'Truck', 'Pedestrian',
           'Person_sitting', 'Cyclist', 'Tram', 'Misc']

print(f'best.pt 존재: {os.path.exists(best_model_path)}')

best.pt 존재: True


In [6]:
def self_drive_assist_yolo(img_path, model, size_limit=300, conf_threshold=0.3):
    """
    YOLOv8 기반 자율주행 보조 시스템
    RetinaNet 버전과 동일한 판단 로직 적용 → 공정한 비교 가능
    
    정지 조건:
      1. Pedestrian / Person_sitting 감지 → Stop
      2. Car / Van / Truck bbox 크기 >= size_limit → Stop
    
    예외 조건:
      3. 이미지 없음 / 열기 실패 → Stop
      4. 탐지 객체 없음 → Stop (불확실)
      5. Cyclist 감지 → Stop
    """
    # 예외 조건 3: 이미지 파일 확인
    if not os.path.exists(img_path):
        return 'Stop - 이미지 파일 없음'
    
    try:
        # YOLOv8 추론
        results = model.predict(
            img_path,
            conf=conf_threshold,
            iou=0.5,
            verbose=False
        )
        result = results[0]
        boxes  = result.boxes
        
        # 예외 조건 4: 탐지 객체 없음
        if boxes is None or len(boxes) == 0:
            return 'Stop - 탐지 객체 없음 (불확실)'
        
        for box in boxes:
            cls        = int(box.cls[0])
            class_name = CLASSES[cls]
            x1, y1, x2, y2 = map(float, box.xyxy[0])
            
            # 정지 조건 1: 사람 감지
            if class_name in ['Pedestrian', 'Person_sitting']:
                return 'Stop - 사람 감지'
            
            # 예외 조건 5: 자전거 탑승자
            if class_name == 'Cyclist':
                return 'Stop - Cyclist 감지'
            
            # 정지 조건 2: 차량 근접 (bbox 크기로 판단)
            box_w = x2 - x1
            box_h = y2 - y1
            if class_name in ['Car', 'Van', 'Truck']:
                if box_w >= size_limit or box_h >= size_limit:
                    return 'Stop - 차량 근접'
        
        return 'Go'
    
    except Exception as e:
        return f'Stop - 오류 발생: {e}'


# 테스트 실행
import os

if os.path.exists(best_model_path):
    best_model  = YOLO(best_model_path)
    test_img    = os.path.join(HOME, 'work/object_detection/data/stop_1.png')
    
    print('=== YOLOv8 자율주행 보조 시스템 테스트 ===')
    result = self_drive_assist_yolo(test_img, best_model)
    print(f'판단 결과: {result}')
else:
    print('학습 완료 후 실행해주세요')

=== YOLOv8 자율주행 보조 시스템 테스트 ===
판단 결과: Stop - 이미지 파일 없음


## STEP 10. 최종 비교 분석 정리

학습 완료 후 아래 표를 채워주세요.

In [8]:
# 측정된 결과값 직접 입력
RETINA_MAP50 = 0.5642   # compute_map()에서 측정된 값
RETINA_ACC   = 0.0      # 정확도 측정값 (있으면 입력)
RETINA_FPS   = 12.01    # FPS 측정값

YOLO_MAP50   = 0.7740   # YOLOv8 val() 결과
YOLO_ACC     = 0.0
YOLO_FPS     = 46.25    # YOLOv8 FPS 측정값

print('결과값 설정 완료 ✅')
print(f'RetinaNet  → mAP: {RETINA_MAP50}, FPS: {RETINA_FPS}')
print(f'YOLOv8s    → mAP: {YOLO_MAP50},   FPS: {YOLO_FPS}')

결과값 설정 완료 ✅
RetinaNet  → mAP: 0.5642, FPS: 12.01
YOLOv8s    → mAP: 0.774,   FPS: 46.25


In [10]:
print('=' * 70)
print('최종 비교 분석 결과')
print('=' * 70)
print(f'{"항목":<25} {"RetinaNet":>20} {"YOLOv8s":>20}')
print('-' * 70)
print(f'{"백본":<25} {"ResNet50":>20} {"CSPDarknet":>20}')
print(f'{"앵커 방식":<25} {"앵커 기반 (9개/위치)":>20} {"앵커 프리":>20}')
print(f'{"Loss":<25} {"Focal + SmoothL1":>20} {"BCE + CIoU":>20}')
print(f'{"mAP@0.5":<25} {str(RETINA_MAP50):>20} {str(YOLO_MAP50):>20}')
print(f'{"FPS":<25} {str(RETINA_FPS):>20} {str(YOLO_FPS):>20}')
print('=' * 70)

최종 비교 분석 결과
항목                                   RetinaNet              YOLOv8s
----------------------------------------------------------------------
백본                                    ResNet50           CSPDarknet
앵커 방식                            앵커 기반 (9개/위치)                앵커 프리
Loss                          Focal + SmoothL1           BCE + CIoU
mAP@0.5                                 0.5642                0.774
FPS                                      12.01                46.25
